In [ ]:
import itertools

import matplotlib.animation
import matplotlib.pyplot as plt
import numpy as np
from scene_synthesis import CircularTrajectory, Rotation, SplineTrajectory, Translation, UniformLinearTrajectory

In [ ]:
# For displaying purposes only
# Some code of this cell was taken from the scipy-documentation:
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.transform.RigidTransform.html#scipy.spatial.transform.RigidTransform
# The copyright of this remains with the SciPy Community and  is subject to the following license:

# Copyright (c) 2001-2002 Enthought, Inc. 2003, SciPy Developers.
# All rights reserved.

# Redistribution and use in source and binary forms, with or without
# modification, are permitted provided that the following conditions
# are met:

# 1. Redistributions of source code must retain the above copyright
#    notice, this list of conditions and the following disclaimer.

# 2. Redistributions in binary form must reproduce the above
#    copyright notice, this list of conditions and the following
#    disclaimer in the documentation and/or other materials provided
#    with the distribution.

# 3. Neither the name of the copyright holder nor the names of its
#    contributors may be used to endorse or promote products derived
#    from this software without specific prior written permission.

# THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS
# "AS IS" AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT
# LIMITED TO, THE IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR
# A PARTICULAR PURPOSE ARE DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT
# OWNER OR CONTRIBUTORS BE LIABLE FOR ANY DIRECT, INDIRECT, INCIDENTAL,
# SPECIAL, EXEMPLARY, OR CONSEQUENTIAL DAMAGES (INCLUDING, BUT NOT
# LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR SERVICES; LOSS OF USE,
# DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER CAUSED AND ON ANY
# THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY, OR TORT
# (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE
# OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.
plt.rcParams["animation.html"] = "jshtml"
plt.rcParams['figure.dpi'] = 150
plt.ioff()

N = 50
t_interval=(-1, 2)

all_times = np.linspace(*t_interval, N)

colors = ("#FF6666", "#005533", "#1199EE")  # Colorblind-safe RGB

def prepare_plot(ax):  # noqa: D103
    for axis, c in zip((ax.xaxis, ax.yaxis, ax.zaxis), colors):  # noqa: B905
        axlabel = axis.axis_name
        axis.set_label_text(axlabel)
        axis.label.set_color(c)
        axis.line.set_color(c)
        axis.set_tick_params(colors=c)

        ax.set(xlim3d=(-10, 10))
        ax.set(ylim3d=(-5, 15))
        ax.set(zlim3d=(-10, 10))

def plot_trajectory(ax, traj, c='black'):  # noqa: D103

        all_xpos = traj.location(all_times)

        def plotter(t):
            ax.plot(*all_xpos, color=c, lw=0.5)
            #ax.plot(xs=all_xpos[:,0], ys=all_xpos[:,1], zs=all_xpos[:,2], c, lw=0.5)

            location = traj.location(t)
            velocity = traj.velocity(t) / 5
            ax.scatter(*location, marker='o', color=c)

            ax.plot(*zip(location, location+velocity), lw=2, color='grey')  # noqa: B905

        return plotter


def plot_reference_frame(ax, ref_frame, name, scale=1):  # noqa: D103

    #loc = np.array([t, t])
    def plotter(tf_t):

        tf = ref_frame.rigid_transform(tf_t)
        tf_t, tf_r = tf.as_components()

        for i, c in enumerate(colors):
            line = np.zeros((2, 3))
            line[1, i] = scale
            line_rot = tf.apply(line)
            line_plot = line_rot #+ loc
            ax.plot(line_plot[:, 0], line_plot[:, 1], line_plot[:, 2], c)
            ax.text(*tf_t, s=name, color="k", va="center", ha="center",
                    bbox={"fc": "w", "alpha": 0.8, "boxstyle": "circle"})
    return plotter

def create_animation(traj_list=None, rf_dict=None):  # noqa: D103

    traj_list = traj_list if traj_list is not None else []
    rf_dict = rf_dict if rf_dict is not None else {}

    t_interval=(-1, 2)
    colors = plt.cm.tab10.colors

    fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

    all_times = np.linspace(*t_interval, 100)
    line_plotter = [plot_trajectory(ax, tr, c) for tr, c in zip(traj_list, itertools.cycle(colors))]
    rf_plotter = [plot_reference_frame(ax, rf, name, 5) for name, rf in rf_dict.items()]

    def animate(t):
        plt.cla()

        prepare_plot(ax)

        [plotter(t) for plotter in line_plotter]
        [plotter(t) for plotter in rf_plotter]


    return matplotlib.animation.FuncAnimation(
        fig, animate, frames=all_times, interval=(all_times[1]-all_times[0]) * 1000
    )

# Representation of Frame of References over Time (FOROT)

## Mathematical Background and Notation

A *Trajectory* is a function $\phi: \mathbb{R}\rightarrow\mathbb{R}^3$, yielding some position vector for every point in time $t \mapsto \mathbf{x}$. Typically we want $\mathbf{x}$ to be expressed in the same *stationary reference frame* as the microphone locations, called here *global reference frame*. $$\mathbf{x} = x_1\mathbf{e}_1 +  x_2\mathbf{e}_2 +  x_3\mathbf{e}_3$$

However, we also want to be able to map a whole set of points that are fixed to some moving body to the global reference frame. 

This is where the *frame of reference over time* comes in. With it, we can express the motion of the whole source region directly without restricting ourselves to a single point. *Any* point with constant coordinates in this FOROT is automatically fixed to whatever we are focussing on.

### Coordinate Transformation
We only consider *Rigid Transformation* (*Isometries*), which consist of a *Rotation* and *Translation* (shift by offset). 
*Rigid Transformation* are appropriate for describing the movement of solid bodies, as they preserve distances and angles.

Usually, a spatial transformation $S: \mathbb{R}^3\rightarrow\mathbb{R}^3$ just maps one point expressed in local coordinates to its counterpart in global coordinates $\mathbf{x_L} \rightarrow \mathbf{x_G}$

In our case this transformation $R$ is not constant, it also depends on time: 
$$(t, \mathbf{x_L})\mapsto\mathbf{x_G}$$
$$R: (\mathbb{R}, \mathbb{R}^3)\rightarrow\mathbb{R}^3$$

This way to express it shows that we can retrieve a trajectory for any point by supplying some point fixed to the frame of reference $\phi_\mathbf{x_L} = R(\cdot, \mathbf{x_L})$. We can also retrieve a normal coordinate transform by supplying a time $S_t = R(t, \cdot)$

*This is not new information and implicitly is build into Acoulars MovingGrid. 
But it does motivate the class structure, which is why it's good to explicitly write it out.*

## Trajectory 

So far, we provide three different types of `Trajectory`s

#### Unform Linear Trajectory
Has constant velocity $\mathbf{v}_0 = \frac{\partial}{\partial t}\phi(t)$ and initial position $\phi(t=0)=\mathbf{x}_0 $

In [ ]:
ult = UniformLinearTrajectory(x0=(0.,5.,0.), v0=(5.,0.,0.))

#### Trajectory through interpolating B-Spline 

In [ ]:
_rp = [[-5, 5, 0], [0, 5, 0], [5, 10, 5], [10, 10, 5]]
spt = SplineTrajectory(times=[-1, -0.2, 1.2, 2], positions=_rp)

#### Circular Trajectory

Circular Trajectory is defined by some initial position and an axis of rotation.
The axis of rotation is defined by some point on it called fixpoint and its direction.

In [ ]:
ct1 = CircularTrajectory(ref_pos=(0.,0.,0), rev_per_sec=1., axis_dir=(1.,0.,0), fixpoint=(0.,0.,5.))
ct2 = CircularTrajectory(ref_pos=(5.,0.,0.), rev_per_sec=1., axis_dir=(1.,1.,1.), fixpoint=(0.,0.,0.))

In [ ]:
# The plots take quite some time to render!
create_animation([ult, spt, ct1, ct2])

## FOROT
For now, only two reference frames (`Translation`, `Rotation`) are implemented.

*For those two classes, the name denotes the part of the transformation which depends on time. E.g. `Translation` is made up of a translation (shift) which varies with time and a constant rotation. `Rotation` is made up of a constant translation (shift) and a rotation around some axis by some frequency.*

Other reference frames could somewhat easily be added. Ideas for future reference frames are:
- a composition of reference frames: This would allow e.g. a rotor that moves through space.
- a rotation that has a nonconstant angular frequency 
- translation where the orientation follows the curve (similar to Acoulars `rvec`)


Note that there no direct mapping $(t, \mathbf{x_L})\rightarrow\mathbf{x_G}$ is required. 
Typically, one only ever needs either to transform
- a single point at many discrete points in time
- or a whole set of points (e.g. the grid) at a single point in time.

This is reflected by the two methods `.trajectory(...)` and `.ref_frame(...)` respectively.
Because of this, both use cases can be implemented efficiently, in most cases without 
slow for loops.

*To map a single point $\mathbf{x_L}$ at some point in time $t$, one can use two ways*
```python
#rf is object of some subtype of LocalReferenceFrame
x_gobal = rf.trajectory(x_local).location(t)
x_gobal = rf.rigid_transform(t).apply(x_local)
```



### Translation

The *rigid transformation* of `Translation` consists of a time dependent shift, which is defined by its `origin_traj`-attribute.  
`origin_traj` defines the path of the origin of the *local reference frame* and can itself be of any subclass of `Trajectory`,
i.e. the reference frame can follow some arbitrary path.

Note that it also applies some rotation (constant over time) first, here called `orientation`. An application is e.g. the definition of some (large) 
aircraft, which moves along a path and has some orientation that does not change in short time frames. 

#### Example

The Translation can be based on any type of Trajectory object,
including UniformLinearTrajectory, SplineTrajectory
and CircularTrajectory.

Here, they all have different orientation_angles, which results in the axis pointing in other directions than the original frame.

In [ ]:
translation_ult = Translation(origin_traj=ult, orientation_angles=(0,0,45))
translation_sp = Translation(origin_traj=spt, orientation_angles=[90,0,90])
translation_ct = Translation(origin_traj=ct2, orientation_angles=[0,0,0])

translation_dict = {
        'ult': translation_ult,
        'sp': translation_sp,
        'ct': translation_ct,
    }
create_animation(rf_dict=translation_dict)

Using the reference frame-objects, the trajectory of any point in that frame can easily be extracted.

For the case of a Translation object, this results in the original trajectory but shifted in space.
Here, we extract the trajectory of the origin $\mathbf{x}_L=(0, 0, 0)^T$ and point $\mathbf{x}_L=(1, 1, 1)^T$


In [ ]:
trs = ( [rf.trajectory((0., 0., 0.)) for rf in translation_dict.values()]
       + [rf.trajectory((1., 1., 1.)) for rf in translation_dict.values()] )

create_animation(traj_list=trs, rf_dict=translation_dict)


### Rotation

Local Reference Frame that rotates around some axis at a constant rate.

It is defined by the rate of rotation (in revolutions per second) and the axis the points revolve around (`'x'`, `'y'` or `'z'`).

If required, an additional constant shift of the origin can be provided as `origin`.

Similar to `Translation`, a constant rotation around Euler angles can be applied. 
This rotation is applied after the rotation due to the revolutions, but before the constant shift is applied.


#### Example

In [ ]:
rot1 = Rotation(1, orientation_angles=(45, 0, 0), origin=(-5, -3, 4), axis='z')
rot2 = Rotation(1.2, orientation_angles=(0, 0, 0), origin=(0, 0, 5), axis='y')

rf_dict={'r1': rot1, 'r2': rot2}

create_animation(rf_dict=rf_dict)


##### Rotor in Wind Tunnel Coordinate System

It is assumed that the $x$-Axis of the global system is in the direction of the
horizontal flow, $y$ is also horizontal and $z$ is vertical.

If the points will be provided as a $xy$-grid (e.g. `StationaryGrid`) and we want
to have a calculation grid in the rotor plane (perpendicular to flow), we need to
map the local $x$-axis to the global $y$-axis, and the local $y$-axis to the global
$z$-axis. This is achieved by first applying a 90° rotation around the
$x$-axis followed by 90° rotation around the global $z$-axis.

The local axis of rotation is the z-axis.

Here `origin=(5.,0.,0.)` denotes the position $(0,0)^T$ gets mapped to,
i.e. that the rotor has an axial position of 5m.

In [ ]:
rotor_frame = Rotation(0.5, orientation_angles=(90, 0, 90), origin=(5., 0., 0.), axis='z')

trs = [rotor_frame.trajectory((x, y, 0.)) for x in np.arange(-8, 9, step=4) for y in np.arange(-8, 9, step=4)]

create_animation(traj_list=trs, rf_dict={'rotor': rotor_frame})